In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [16]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b")

In [17]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [18]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [19]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [20]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [21]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with “crust” experience and said, “I’m a real slice of life!”',
 'explanation': ''}

In [22]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with “crust” experience and said, “I’m a real slice of life!”', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf7-1d7d-644d-8002-2448bb75f9c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-15T06:34:03.988487+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf6-fef8-6951-8001-a382bf93183e'}}, tasks=(), interrupts=())

In [23]:
list(workflow.get_state_history(config1)) #intermediate state values

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with “crust” experience and said, “I’m a real slice of life!”', 'explanation': ''}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf7-1d7d-644d-8002-2448bb75f9c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-15T06:34:03.988487+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf6-fef8-6951-8001-a382bf93183e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with “crust” experience and said, “I’m a real slice of life!”'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf6-fef8-6951-8001-a382bf93183e'}}, metadata={'source': 'loo

In [24]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?  \nBecause it had too many *nood*ling issues!',
 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | How it’s used in the joke |\n|---------|------------|---------------------------|\n| **Spaghetti** | A type of pasta that is literally made of *noodles* | The subject of the joke; it’s anthropomorphised (given human feelings). |\n| **Therapy** | A professional setting where people work through mental or emotional problems | The “place” the spaghetti goes to, implying it has psychological issues. |\n| **Noodling** | 1. *Noodling* is a verb meaning “to fiddle or play with something” (often used for playing guitar). 2. It’s also a playful, informal way of saying “to be a bit indecisive or absent‑minded.” | The punchline replaces a normal phrase (“too many *nodding* issues” or “too many *mental* issues”) with “noodling.” Because spaghetti is made of noodles, “noodling issues” sounds like “is

In [25]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy?  \nBecause it had too many *nood*ling issues!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | How it’s used in the joke |\n|---------|------------|---------------------------|\n| **Spaghetti** | A type of pasta that is literally made of *noodles* | The subject of the joke; it’s anthropomorphised (given human feelings). |\n| **Therapy** | A professional setting where people work through mental or emotional problems | The “place” the spaghetti goes to, implying it has psychological issues. |\n| **Noodling** | 1. *Noodling* is a verb meaning “to fiddle or play with something” (often used for playing guitar). 2. It’s also a playful, informal way of saying “to be a bit indecisive or absent‑minded.” | The punchline replaces a normal phrase (“too many *nodding* issues” or “too many *mental* issues”) with “noodling.” Because spaghetti is made of noodles, “noodling issu

In [26]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy?  \nBecause it had too many *nood*ling issues!', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | How it’s used in the joke |\n|---------|------------|---------------------------|\n| **Spaghetti** | A type of pasta that is literally made of *noodles* | The subject of the joke; it’s anthropomorphised (given human feelings). |\n| **Therapy** | A professional setting where people work through mental or emotional problems | The “place” the spaghetti goes to, implying it has psychological issues. |\n| **Noodling** | 1. *Noodling* is a verb meaning “to fiddle or play with something” (often used for playing guitar). 2. It’s also a playful, informal way of saying “to be a bit indecisive or absent‑minded.” | The punchline replaces a normal phrase (“too many *nodding* issues” or “too many *mental* issues”) with “noodling.” Because spaghetti is made of noodles, “noodling iss

### Time travel


In [29]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b0cf6-f0a7-649a-8000-63d752745a79"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b0cf6-f0a7-649a-8000-63d752745a79'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-15T06:33:59.287106+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf6-f0a3-6bcf-bfff-3dfb35be4858'}}, tasks=(PregelTask(id='e3e68197-0e91-834c-c341-1bab71d39134', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?\n\nBecause it heard the company was looking for someone with “crust” experience and said, “I’m a real slice of life!”'}),), interrupts=())

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1b0cf6-f0a7-649a-8000-63d752745a79"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it kneaded the dough!',
 'explanation': '**Explanation of the joke**\n\n- **Wordplay on “kneaded” vs. “needed.”**  \n  The sentence “It kneaded the dough” sounds exactly like “It needed the dough.” In everyday speech, “kneaded” is the past tense of “knead,” which is what you do to pizza dough when preparing it. In the joke, it’s used as a pun on “needed,” implying the pizza wanted something.\n\n- **Dual meaning of “dough.”**  \n  *Dough* can mean:  \n  1. The raw, pliable mixture that becomes pizza.  \n  2. Slang for money (“I’m short on dough”).  \n\n  So the joke suggests the pizza *wanted* money (to pay for a job) while also literally needing the dough to be made into a pizza.\n\n- **Putting it together:**  \n  The joke’s humor comes from the double entendre: a pizza “applies for a job” because it “kneaded the dough” (both literally kneading the dough to make pizza and figuratively needing money). The pun rel

In [31]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it kneaded the dough!', 'explanation': '**Explanation of the joke**\n\n- **Wordplay on “kneaded” vs. “needed.”**  \n  The sentence “It kneaded the dough” sounds exactly like “It needed the dough.” In everyday speech, “kneaded” is the past tense of “knead,” which is what you do to pizza dough when preparing it. In the joke, it’s used as a pun on “needed,” implying the pizza wanted something.\n\n- **Dual meaning of “dough.”**  \n  *Dough* can mean:  \n  1. The raw, pliable mixture that becomes pizza.  \n  2. Slang for money (“I’m short on dough”).  \n\n  So the joke suggests the pizza *wanted* money (to pay for a job) while also literally needing the dough to be made into a pizza.\n\n- **Putting it together:**  \n  The joke’s humor comes from the double entendre: a pizza “applies for a job” because it “kneaded the dough” (both literally kneading the dough to make pizza and figuratively needing

### Updating state

In [32]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b0cf6-f0a7-649a-8000-63d752745a79", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b0cfb-3732-6458-8001-7faa0516ef0d'}}

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cfb-3732-6458-8001-7faa0516ef0d'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-15T06:35:54.058248+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b0cf6-f0a7-649a-8000-63d752745a79'}}, tasks=(PregelTask(id='9dbd6aa6-493f-e801-35c9-d504c334de31', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it kneaded the dough!', 'explanation': '**Explanation of the joke**\n\n- **Wordplay on “kneaded” vs. “needed.”**  \n  The sentence “It kneaded the dough” sounds exactly like “It needed the dough.” In everyday speech, “kneaded” is the past tense of “knead,” which is what you do to piz

In [34]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1b0cfb-3732-6458-8001-7faa0516ef0d"}})

{'topic': 'samosa',
 'joke': "Why did the samosa go to therapy?\n\nBecause it had too many *crust*‑worthy secrets and couldn't *stuff* them in silence!",
 'explanation': '**The joke in a nutshell**\n\n> *“Why did the samosa go to therapy?  \n>  Because it had too many *crust‑worthy secrets and couldn’t *stuff* them in silence!”*\n\nThe humor comes from a play on words that ties the physical characteristics of a samosa to the idea of having “secrets” that need talking out. Let’s break it down.\n\n---\n\n### 1. What a samosa is\n\n- **A pastry** that is *wrapped* (or “crusted”) around a filling.\n- The outer shell is the “crust,” and the inner part is the “stuffing” (the “stuff”).\n\n---\n\n### 2. The double meanings\n\n| Word | Literal meaning in a samosa | Figurative / punny meaning |\n|------|------------------------------|-----------------------------|\n| **Crust‑worthy** | “Crust‑worthy” sounds like “crust‑worthy” → something that deserves a crust. | A pun on **“crusty”** or “crust‑

In [35]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'samosa', 'joke': "Why did the samosa go to therapy?\n\nBecause it had too many *crust*‑worthy secrets and couldn't *stuff* them in silence!", 'explanation': '**The joke in a nutshell**\n\n> *“Why did the samosa go to therapy?  \n>  Because it had too many *crust‑worthy secrets and couldn’t *stuff* them in silence!”*\n\nThe humor comes from a play on words that ties the physical characteristics of a samosa to the idea of having “secrets” that need talking out. Let’s break it down.\n\n---\n\n### 1. What a samosa is\n\n- **A pastry** that is *wrapped* (or “crusted”) around a filling.\n- The outer shell is the “crust,” and the inner part is the “stuffing” (the “stuff”).\n\n---\n\n### 2. The double meanings\n\n| Word | Literal meaning in a samosa | Figurative / punny meaning |\n|------|------------------------------|-----------------------------|\n| **Crust‑worthy** | “Crust‑worthy” sounds like “crust‑worthy” → something that deserves a crust. | A pun on **“

### Fault tolerance

In [36]:
import time
from typing import TypedDict

In [37]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [38]:
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [39]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


EmptyInputError: Received no input for __start__

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[]